# OccamRoute — Notebook 01: Aprendizado do Roteador (`SWE-bench Lite` + Scikit-learn & PyTorch)

> **Missão:** Fazer o OccamRoute **aprender** a partir do benchmark real `princeton-nlp/SWE-bench_Lite` e da topologia AST dos repositórios, treinando e calibrando dois roteadores complementares (**Scikit-learn Calibrated GBDT** e **PyTorch Tabular MLP**) para decidir entre modelo leve local e modelo forte de fronteira.

## Fluxo deste Notebook
```text
princeton-nlp/SWE-bench_Lite ──▶ Ground Truth de Patches (arquivos, hunks, churn)
                                        │
RepoGraph (AST + NetworkX) ──▶ SurgicalContextSlicer ──▶ 17 Features Topológicas
                                        │
                        RouteLLM Data Augmentation
                                        │
           ┌────────────────────────────┴────────────────────────────┐
           ▼                                                         ▼
SklearnComplexityRouter (.joblib)                       TorchComplexityRouter (.pt)
(StandardScaler + GBDT + Sigmoid Calib.)                (LayerNorm + SiLU + Dropout MLP)
           └────────────────────────────┬────────────────────────────┘
                                        ▼
             Varredura RouteLLM tau in [0.1, 0.9] -> PGR (%) & Economia de Custo
```

In [ ]:
# @title 1. Setup do Ambiente (Colab / Local) e Imports do `occamroute`
import os
import platform
import sys
from pathlib import Path

!pip install -q datasets networkx numpy scipy tiktoken pandas scikit-learn joblib torch matplotlib seaborn

for candidate in [
    Path("src"),
    Path("../src"),
    Path("/content/OccamRoute/src"),
    Path("/content/src"),
]:
    if (candidate / "occamroute").exists() and str(candidate.resolve()) not in sys.path:
        sys.path.insert(0, str(candidate.resolve()))
        break

import torch
import occamroute
from occamroute import (
    FEATURE_NAMES,
    OccamRouter,
    RepoGraphBuilder,
    SklearnComplexityRouter,
    SubgraphFeatureExtractor,
    SurgicalContextSlicer,
    TorchComplexityRouter,
    TorchRouterConfig,
    build_routing_dataset,
    count_patch_churn,
    extract_ground_truth_files,
    extract_patch_line_ranges,
    load_swebench_lite,
    tasks_to_dataframe,
)

print(f"Python     : {platform.python_version()}")
print(f"OccamRoute : {Path(occamroute.__file__).parent}")
print(f"GPU CUDA   : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Features   : {len(FEATURE_NAMES)} dimensões -> {FEATURE_NAMES}")

In [ ]:
# @title 2. Carregamento do `SWE-bench Lite` e Rotulagem Empírica de Dificuldade do Patch
import pandas as pd

try:
    swe_tasks = load_swebench_lite(limit=40, split="test")
    source_mode = "huggingface:princeton-nlp/SWE-bench_Lite"
except Exception as exc:
    print(f"Aviso: falha ao baixar SWE-bench Lite ({exc}); usando fallback offline mínimo.")
    from occamroute.dataset_loader import SWEBenchTask
    sample_patch = (
        "diff --git a/src/occamroute/context_slicer.py b/src/occamroute/context_slicer.py\n"
        "--- a/src/occamroute/context_slicer.py\n"
        "+++ b/src/occamroute/context_slicer.py\n"
        "@@ -10,3 +10,5 @@ def position_aware_order(ranked_ids):\n"
        "+    if not ranked_ids:\n"
        "+        return []\n"
    )
    swe_tasks = [
        SWEBenchTask(
            instance_id="local__occamroute-001",
            repo="occamroute/local",
            base_commit="HEAD",
            problem_statement="Handle empty list in `position_aware_order`.",
            patch=sample_patch,
            ground_truth_files=extract_ground_truth_files(sample_patch),
            ground_truth_hunks=extract_patch_line_ranges(sample_patch),
            patch_added_lines=2,
            patch_removed_lines=0,
            num_gt_files=1,
        )
    ]
    source_mode = "offline_fallback"

df_swe = tasks_to_dataframe(swe_tasks)
df_swe["total_churn"] = df_swe["patch_added_lines"] + df_swe["patch_removed_lines"]
df_swe["num_hunks"] = [sum(len(v) for v in t.ground_truth_hunks.values()) for t in swe_tasks]
# Oráculo empírico de complexidade do patch: >1 arquivo OU >=15 linhas alteradas OU >=3 hunks => Strong (1)
df_swe["oracle_label"] = (
    (df_swe["num_gt_files"] > 1) | (df_swe["total_churn"] >= 15) | (df_swe["num_hunks"] >= 3)
).astype(int)

print(f"Fonte: {source_mode} | Tarefas: {len(df_swe)} | Strong ratio: {df_swe['oracle_label'].mean():.1%}")
df_swe[["instance_id", "repo", "num_gt_files", "num_hunks", "total_churn", "oracle_label"]].head(12)

In [ ]:
# @title 3. Extração de Grafo AST + Subgrafo Cirúrgico + Dataset Tabular (RouteLLM Augmentation)
import numpy as np

# Indexa `src/occamroute` e opcionalmente um repo real clonado para enriquecer âncoras estruturais
pkg_candidates = [Path("src/occamroute"), Path("../src/occamroute"), Path("/content/OccamRoute/src/occamroute")]
pkg_dir = next((p.resolve() for p in pkg_candidates if p.exists()), Path(occamroute.__file__).parent)

repo_graph = RepoGraphBuilder().build_from_directory(pkg_dir)
print("RepoGraph Stats:", repo_graph.summary_stats())

dataset = build_routing_dataset(repo_graph=repo_graph, n_samples=480, val_ratio=0.25, seed=42)
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

df_train = pd.DataFrame(dataset.X_train, columns=dataset.feature_names)
df_train["label"] = dataset.y_train
df_train.to_csv(artifacts_dir / "router_training_dataset.csv", index=False)

print(f"Dataset de Treino : X_train={dataset.X_train.shape} | positive_rate={dataset.y_train.mean():.2f}")
print(f"Dataset Validação : X_val={dataset.X_val.shape}   | positive_rate={dataset.y_val.mean():.2f}")

In [ ]:
# @title 4. Análise de Separabilidade das 17 Features Topológicas (Weak vs. Strong)
summary_rows = []
for idx, fname in enumerate(dataset.feature_names):
    vals_0 = dataset.X_train[dataset.y_train == 0, idx]
    vals_1 = dataset.X_train[dataset.y_train == 1, idx]
    pooled_std = float(np.sqrt(0.5 * (np.var(vals_0) + np.var(vals_1))) + 1e-6)
    effect_d = (float(np.mean(vals_1)) - float(np.mean(vals_0))) / pooled_std
    summary_rows.append({
        "feature": fname,
        "weak_mean": float(np.mean(vals_0)),
        "strong_mean": float(np.mean(vals_1)),
        "cohen_d": effect_d,
    })

df_sep = pd.DataFrame(summary_rows).sort_values("cohen_d", key=lambda s: s.abs(), ascending=False)
print(df_sep.round(3).to_string(index=False))

In [ ]:
# @title 5. Treinamento & Calibração: `SklearnComplexityRouter` (GBDT + CalibratedClassifierCV)
from sklearn.inspection import permutation_importance

sk_router = SklearnComplexityRouter().fit(dataset.X_train, dataset.y_train)
sk_metrics = sk_router.evaluate(dataset.X_val, dataset.y_val, threshold=0.50)
sk_path = sk_router.save(artifacts_dir / "sklearn_router.joblib")

perm = permutation_importance(
    sk_router.pipeline,
    dataset.X_val,
    dataset.y_val,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc",
)
df_imp = pd.DataFrame({
    "feature": list(dataset.feature_names),
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=False)

print("Sklearn Metrics (val):", {k: round(v, 4) for k, v in sk_metrics.items()})
print(f"Salvo em: {sk_path}\n")
print("Top 8 features (Permutation ROC-AUC):")
print(df_imp.head(8).round(4).to_string(index=False))

In [ ]:
# @title 6. Treinamento Neural: `TorchComplexityRouter` (`TabularRouterMLP` em PyTorch)
torch_cfg = TorchRouterConfig(
    input_dim=len(dataset.feature_names),
    hidden_dims=(32, 16),
    dropout=0.15,
    lr=3e-3,
    epochs=120,
    batch_size=32,
    seed=42,
)
pt_router = TorchComplexityRouter(config=torch_cfg).fit(dataset.X_train, dataset.y_train)
pt_metrics = pt_router.evaluate(dataset.X_val, dataset.y_val, threshold=0.50)
pt_path = pt_router.save(artifacts_dir / "torch_router.pt")

print("PyTorch MLP Metrics (val):", {k: round(v, 4) for k, v in pt_metrics.items()})
print(f"Loss inicial -> final: {pt_router.train_losses[0]:.4f} -> {pt_router.train_losses[-1]:.4f}")
print(f"Salvo em: {pt_path}")

In [ ]:
# @title 7. Varredura RouteLLM (tau in [0.1, 0.9]) e Seleção do Ponto Ótimo
import json

sk_engine = OccamRouter.from_artifact(sk_path, threshold=0.50)
pt_engine = OccamRouter.from_artifact(pt_path, threshold=0.50)

sk_sweep = pd.DataFrame(
    sk_engine.sweep_thresholds(dataset.X_val, dataset.y_val, dataset.weak_quality_val, dataset.strong_quality_val)
)
pt_sweep = pd.DataFrame(
    pt_engine.sweep_thresholds(dataset.X_val, dataset.y_val, dataset.weak_quality_val, dataset.strong_quality_val)
)

feasible = pt_sweep[pt_sweep["cost_savings_pct"] >= 35.0]
best_row = (feasible if not feasible.empty else pt_sweep).sort_values("pgr_pct", ascending=False).iloc[0]
optimal_tau = float(best_row["threshold"])

(artifacts_dir / "router_metrics.json").write_text(
    json.dumps(
        {
            "optimal_tau": optimal_tau,
            "sklearn": sk_metrics,
            "torch": pt_metrics,
            "sklearn_sweep": sk_sweep.to_dict(orient="records"),
            "torch_sweep": pt_sweep.to_dict(orient="records"),
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Ponto ótimo RouteLLM (PyTorch): tau*={optimal_tau:.1f} | PGR={best_row['pgr_pct']:.1f}% | Economia={best_row['cost_savings_pct']:.1f}%\n")
print(pt_sweep.round(3).to_string(index=False))

In [ ]:
# @title 8. Diagnóstico Visual: ROC Curves, Calibração e Convergência MLP (Dark Theme)
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve

Path("assets").mkdir(exist_ok=True)
BG_FIG, BG_AX, TEXT_MAIN, TEXT_MUTED, GRID_COL = "#0D1117", "#161B22", "#E6EDF3", "#8B949E", "#30363D"

plt.rcParams.update({
    "figure.facecolor": BG_FIG,
    "axes.facecolor": BG_AX,
    "axes.edgecolor": GRID_COL,
    "axes.labelcolor": TEXT_MAIN,
    "xtick.color": TEXT_MAIN,
    "ytick.color": TEXT_MUTED,
    "text.color": TEXT_MAIN,
})

sk_probs = sk_router.predict_proba(dataset.X_val)
pt_probs = pt_router.predict_proba(dataset.X_val)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.8), dpi=250)
fig.patch.set_facecolor(BG_FIG)

# 1. ROC Curves
fpr_sk, tpr_sk, _ = roc_curve(dataset.y_val, sk_probs)
fpr_pt, tpr_pt, _ = roc_curve(dataset.y_val, pt_probs)
ax1.plot(fpr_sk, tpr_sk, color="#58A6FF", lw=2.0, label=f"Sklearn GBDT (AUC={sk_metrics['roc_auc']:.3f})")
ax1.plot(fpr_pt, tpr_pt, color="#2EA043", lw=2.0, label=f"PyTorch MLP (AUC={pt_metrics['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], color=GRID_COL, linestyle="--", lw=1.0)
ax1.set_title("ROC Curve (Validation)", fontsize=11.5, fontweight="bold")
ax1.set_xlabel("False Positive Rate")
ax1.set_ylabel("True Positive Rate")
ax1.grid(True, color=GRID_COL, ls="--", alpha=0.5)
ax1.legend(facecolor=BG_FIG, edgecolor=GRID_COL, fontsize=8.5, loc="lower right")

# 2. Calibration Diagram
pt_true, pt_pred = calibration_curve(dataset.y_val, pt_probs, n_bins=8, strategy="uniform")
sk_true, sk_pred = calibration_curve(dataset.y_val, sk_probs, n_bins=8, strategy="uniform")
ax2.plot(sk_pred, sk_true, marker="o", color="#58A6FF", lw=1.8, label=f"Sklearn (ECE={sk_metrics['ece']:.3f})")
ax2.plot(pt_pred, pt_true, marker="s", color="#2EA043", lw=1.8, label=f"PyTorch (ECE={pt_metrics['ece']:.3f})")
ax2.plot([0, 1], [0, 1], color=GRID_COL, linestyle="--", lw=1.0)
ax2.set_title("Probability Calibration", fontsize=11.5, fontweight="bold")
ax2.set_xlabel("Mean Predicted P(strong)")
ax2.set_ylabel("Empirical Fraction Strong")
ax2.grid(True, color=GRID_COL, ls="--", alpha=0.5)
ax2.legend(facecolor=BG_FIG, edgecolor=GRID_COL, fontsize=8.5, loc="upper left")

# 3. PyTorch Loss Curve
ax3.plot(range(1, len(pt_router.train_losses) + 1), pt_router.train_losses, color="#A371F7", lw=2.0)
ax3.set_title("PyTorch MLP Training Convergence", fontsize=11.5, fontweight="bold")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("BCEWithLogits Loss")
ax3.grid(True, color=GRID_COL, ls="--", alpha=0.5)

for ax in (ax1, ax2, ax3):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("OccamRoute Phase 2 — Router Learning & Calibration Diagnostics", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
diag_path = Path("assets/router_learning_diagnostics.png")
fig.savefig(diag_path, dpi=250, bbox_inches="tight", facecolor=BG_FIG)
plt.show()
print(f"Salvo: {diag_path}")

In [ ]:
# @title 9. Importância das Features Topológicas e Curva PGR x Economia (RouteLLM)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 5.2), dpi=250)
fig.patch.set_facecolor(BG_FIG)

top_imp = df_imp.head(10).iloc[::-1]
ax1.barh(top_imp["feature"], top_imp["importance"], color="#58A6FF", edgecolor="#0D1117")
ax1.set_title("Top 10 Topological Features (Permutation ROC-AUC)", fontsize=11.5, fontweight="bold")
ax1.set_xlabel("Mean ROC-AUC Drop on Permutation")
ax1.grid(axis="x", color=GRID_COL, ls="--", alpha=0.5)
ax1.set_axisbelow(True)

ax2.plot(sk_sweep["cost_savings_pct"], sk_sweep["pgr_pct"], marker="o", color="#58A6FF", lw=2.0, label="Sklearn Calibrated GBDT")
ax2.plot(pt_sweep["cost_savings_pct"], pt_sweep["pgr_pct"], marker="D", color="#2EA043", lw=2.2, label="PyTorch Tabular MLP")
for _, r in pt_sweep.iterrows():
    ax2.annotate(f"t={r['threshold']:.1f}", (r["cost_savings_pct"], r["pgr_pct"]), xytext=(4, 4), textcoords="offset points", fontsize=8, color=TEXT_MUTED)

ax2.set_title("RouteLLM Trade-off: Cost Savings vs. PGR (%)", fontsize=11.5, fontweight="bold")
ax2.set_xlabel("Cost Savings vs. Always-Strong (%)")
ax2.set_ylabel("Performance Gap Recovered - PGR (%)")
ax2.grid(True, color=GRID_COL, ls="--", alpha=0.5)
ax2.legend(facecolor=BG_FIG, edgecolor=GRID_COL, fontsize=8.8, loc="lower left")

for ax in (ax1, ax2):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
pgr_path = Path("assets/router_feature_and_pgr_sweep.png")
fig.savefig(pgr_path, dpi=250, bbox_inches="tight", facecolor=BG_FIG)
plt.show()
print(f"Salvo: {pgr_path}")

In [ ]:
# @title 10. Teste Ao Vivo com os Artefatos Salvos (`OccamRouter.from_artifact`)
slicer = SurgicalContextSlicer(repo_graph)
extractor = SubgraphFeatureExtractor(repo_graph)

loaded_sk = OccamRouter.from_artifact(artifacts_dir / "sklearn_router.joblib", threshold=optimal_tau)
loaded_pt = OccamRouter.from_artifact(artifacts_dir / "torch_router.pt", threshold=optimal_tau)

unseen_queries = [
    "Explain what `position_aware_order` returns for empty lists.",
    "What does `extract_ground_truth_files` parse from git diff headers?",
    "Trace cross-module dependencies from `RepoGraphBuilder.build_from_directory` through `SurgicalContextSlicer.slice` and `SubgraphFeatureExtractor.extract`.",
    "Refactor `SklearnComplexityRouter` and `TorchComplexityRouter` calibration and `OccamRouter.sweep_thresholds`.",
]

for idx, q in enumerate(unseen_queries, start=1):
    sl, dec_pt = loaded_pt.route_query(q, slicer, extractor, token_budget=1500, max_symbols=6)
    dec_sk = loaded_sk.route_slice(sl, extractor)
    print(
        f"[Q{idx}] tokens={sl.context_tokens:<4} nodes={len(sl.ordered_node_ids)} | "
        f"PyTorch: {dec_pt.selected_tier:<15} (P={dec_pt.p_strong:.3f}) | "
        f"Sklearn: {dec_sk.selected_tier:<15} (P={dec_sk.p_strong:.3f})"
    )